In [2]:
!pip install transformers torch torchvision pillow faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 62.5 MB/s eta 0:00:00


Data preprocessing

In [1]:
import torch
import numpy as np
from PIL import Image
from pathlib import Path

from transformers import CLIPProcessor, CLIPModel
import torch.nn.functional as F
import faiss

In [3]:
!apt-get install unrar -y
!unrar x /content/Desktop.rar /content/extracted/

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/Desktop.rar

Creating    /content/extracted                                        OK
Creating    /content/extracted/images                                 OK
Extracting  /content/extracted/images/bird_1.jfif                          4%  OK 
Extracting  /content/extracted/images/car_1.jfif                          14%  OK 
Extracting  /content/extracted/images/key_1.jfif                          15%  OK 
Extracting  /content/extracted/images/pet_1.jfif                          22%  OK 
Extracting  /content/extracted/images/phone_1.jfif                        25%  OK 
Extracting  /content/extracted/images/walet_1.jfif           

In [4]:
import os

for root, dirs, files in os.walk("/content/extracted"):
    print(root)
    for f in files:
        print("  ", f)

/content/extracted
/content/extracted/images
   phone_1.jfif
   car_1.jfif
   pet_1.jfif
   bird_1.jfif
   walet_1.jfif
   key_1.jfif
/content/extracted/imges1
   Bird_2.jfif
   phone_2.jpg
   pet_2.jfif
   car_2.jfif
   walet_2.jfif
   key_2.jfif


Load Images

In [5]:
from PIL import Image
from pathlib import Path

def load_all_images(base_path):
    base_path = Path(base_path)
    images = []
    paths = []

    exts = ["*.jpg", "*.jpeg", "*.png", "*.jfif", "*.webp"]

    for ext in exts:
        for img_path in base_path.rglob(ext):  # 🔥 مهم: rglob یعنی همه فولدرها
            try:
                img = Image.open(img_path).convert("RGB")
                images.append(img)
                paths.append(str(img_path))
            except:
                pass

    return images, paths

In [6]:
images, image_paths = load_all_images("/content/extracted")
print("Total images:", len(images))

Total images: 12


Embeding

In [44]:
from transformers import AutoProcessor, SiglipVisionModel

processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
model = SiglipVisionModel.from_pretrained("google/siglip-base-patch16-224")

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.sel

In [46]:
inputs = processor(images=images, return_tensors="pt")
outputs = model(**inputs)

features = outputs.pooler_output

In [47]:
import torch.nn.functional as F

features = F.normalize(features, p=2, dim=-1)

In [48]:
inputs = processor(images=images, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    features = outputs.pooler_output

features = torch.nn.functional.normalize(features, dim=-1)

In [52]:
image_embeddings = compute_embeddings(images, model, processor)

In [53]:
image_embeddings

[array([-1.41940583e-02, -1.16002811e-02, -3.08143776e-02,  2.44147815e-02,
         1.51040815e-02, -8.49110167e-03, -1.50385173e-03, -1.60693889e-03,
        -2.04832852e-02,  4.45426106e-02,  1.85250752e-02,  2.95440536e-02,
        -2.13096794e-02, -1.65844578e-02,  7.45577514e-02,  4.32669334e-02,
         1.89864412e-02,  3.43541689e-02,  2.30624005e-02, -5.63558303e-02,
         6.19184896e-02, -2.09869929e-02,  1.31690390e-02,  2.11051125e-02,
         1.27258869e-02, -4.58223978e-03, -1.29693020e-02,  3.23339831e-03,
         9.54016112e-03, -3.66595834e-02,  6.36889972e-03, -5.69003681e-03,
         8.03986192e-03,  1.36010684e-02, -2.89280992e-02, -3.55545036e-03,
         9.24062449e-03,  1.00256130e-02,  1.60183106e-02, -2.31437460e-02,
        -3.64319459e-02,  2.16579512e-02,  1.81022729e-03, -1.37953656e-02,
         3.09086051e-02,  2.26108562e-02,  3.14448103e-02, -1.88990720e-02,
        -4.83022220e-02, -9.47167575e-02,  2.27054255e-03,  4.64495420e-02,
        -9.4

In [54]:
query_embedding = image_embeddings[0]

In [56]:
gallery_embeddings = image_embeddings
query_embedding = image_embeddings[0]

In [57]:
import torch
import torch.nn.functional as F

def find_similar(query_embedding, gallery_embeddings, top_k=3):
    gallery_embeddings = torch.tensor(gallery_embeddings)

    query_embedding = torch.tensor(query_embedding)

    sims = F.cosine_similarity(
        query_embedding.unsqueeze(0),
        gallery_embeddings
    )

    top_k_idx = sims.topk(top_k).indices
    return top_k_idx

In [58]:
top_idx = find_similar(query_embedding, image_embeddings, top_k=3)
print(top_idx)

tensor([ 0,  5, 10])


/tmp/ipykernel_5220/3346564025.py:5: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  gallery_embeddings = torch.tensor(gallery_embeddings)
